In [22]:
# !pip uninstall -y scikit-learn scikit-survival
# !pip install --no-cache-dir scikit-learn==1.3.2
# !pip install --no-cache-dir scikit-survival==0.22.2
# !pip install --no-cache-dir lifelines

In [23]:
# import sksurv
# import sklearn
# import lifelines

# print("OK")

In [24]:
# !pip install pycox torchtuples
# !pip install lifelines

In [25]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import KFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from scipy.stats import rankdata
from sklearn.impute import SimpleImputer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.preprocessing import OneHotEncoder as SKSurvOneHotEncoder
from pycox.models import CoxPH
from pycox.models import CoxTime
from torchtuples.practical import MLPVanilla
from torchtuples import optim as ttoptim
import torch
from functools import partial
import optuna
from pycox.models import DeepHitSingle
from pycox.models.loss import CoxPHLoss
from pycox.models import DeepHitSingle
from pathlib import Path

In [26]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/equity-post-HCT-survival-predictions/sample_submission.csv
/kaggle/input/competitions/equity-post-HCT-survival-predictions/data_dictionary.csv
/kaggle/input/competitions/equity-post-HCT-survival-predictions/train.csv
/kaggle/input/competitions/equity-post-HCT-survival-predictions/test.csv


In [27]:
train = pd.read_csv("/kaggle/input/competitions/equity-post-HCT-survival-predictions/train.csv")
test = pd.read_csv("/kaggle/input/competitions/equity-post-HCT-survival-predictions/test.csv")
sub_path = Path('/kaggle/input/competitions/equity-post-HCT-survival-predictions/sample_submission.csv')
print(train.head())

   ID                       dri_score psych_disturb    cyto_score diabetes  \
0   0  N/A - non-malignant indication            No           NaN       No   
1   1                    Intermediate            No  Intermediate       No   
2   2  N/A - non-malignant indication            No           NaN       No   
3   3                            High            No  Intermediate       No   
4   4                            High            No           NaN       No   

   hla_match_c_high  hla_high_res_8          tbi_status arrhythmia  \
0               NaN             NaN              No TBI         No   
1               2.0             8.0  TBI +- Other, >cGy         No   
2               2.0             8.0              No TBI         No   
3               2.0             8.0              No TBI         No   
4               2.0             8.0              No TBI         No   

   hla_low_res_6  ...          tce_div_match donor_related  \
0            6.0  ...                    NaN    

In [28]:
def missing_vals(df):
    """prints out columns with perc of missing values"""
    missing = [
        (df.columns[idx], perc)
        for idx, perc in enumerate(df.isna().mean() * 100)
        if perc > 0
    ]

    if len(missing) == 0:
        return "no missing values"


    missing.sort(key=lambda x: x[1], reverse=True)

    print(f"There are a total of {len(missing)} variables with missing values\n")

    for tup in missing:
        print(str.ljust(f"{tup[0]:<20} => {round(tup[1], 3)}%", 1))

In [29]:
def compute_survival_probability(df, time_col='efs_time', event_col='efs'):
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    return kmf.survival_function_at_times(df[time_col]).values

train['y'] = compute_survival_probability(train, time_col='efs_time', event_col='efs')
train['efs_time_new'] = train['efs_time'].copy()
train.loc[train['efs'] == 0, 'efs_time_new'] *= -1

In [30]:
id_column = 'ID'

drop_features = ['ID', 'efs', 'efs_time', 'y', 'efs_time_new']
features = [col for col in train.columns if col not in drop_features]
numeric_features = train.select_dtypes(include=['float64', 'int64']).columns.intersection(features).tolist()
categorical_features = train.select_dtypes(include=['object', 'category']).columns.intersection(features).tolist()

features_train = pd.DataFrame(train[features])

features_test = pd.DataFrame(test[features])

print(features_test.shape)
print(features_train.shape)

(3, 57)
(28800, 57)


In [31]:
num_imputer = SimpleImputer(strategy='median')
for col in categorical_features:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

numeric_transformer = Pipeline(steps=[
    ('imputer', num_imputer),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
X_train = preprocessor.fit_transform(features_train)
X_test = preprocessor.transform(features_test)
y_train = train['y']

In [32]:
n_folds = 15
KF = KFold(n_splits = n_folds, shuffle = True, random_state = 42)

def stratified_concordance(solution: pd.DataFrame, submission: pd.DataFrame, id_column: str) -> float:
    if id_column in solution.columns:
        solution = solution.drop(columns=[id_column])
    if id_column in submission.columns:
        submission = submission.drop(columns=[id_column])

    event_label = 'efs'
    interval_label = 'efs_time'
    prediction_label = 'prediction'

    merged_df = pd.concat([solution, submission], axis=1)
    merged_df.reset_index(inplace=True)
    merged_df.dropna(subset=[event_label, interval_label, prediction_label], inplace=True)
    groups = dict(merged_df.groupby(['race_group']).groups)

    scores = []
    for group in groups.keys():
        indices = sorted(groups[group])
        subset = merged_df.iloc[indices]
        # if len(subset) < 2 or subset[event_label].nunique() < 2 or subset[interval_label].nunique() < 2:
        #     print(f"Warning: Skipping group '{group}' due to insufficient data for concordance calculation.")
        #     continue
        try:
            concordance = concordance_index(
                subset[interval_label],
                -subset[prediction_label],
                subset[event_label]
            )
            scores.append(concordance)
        except ZeroDivisionError:
            print(f"Warning: Skipping group '{group}' due to insufficient data for concordance calculation.")
            continue

    if not scores:
        return 0.5
    return float(np.mean(scores) - np.sqrt(np.var(scores)))

In [33]:
def model_evaluation(models, X, y, X_test, train_df, id_column, folds = n_folds, seed=42):
    oof_predictions = [np.zeros((len(X), 1)) for _ in models]
    model_scores = [[[] for _ in range(folds)] for _ in models]
    final_predictions = [np.zeros(len(X_test)) for _ in models]

    kf = KFold(n_splits=folds, shuffle=True, random_state=seed)

    for model_idx, model in enumerate(models):
        print(f"\n{'='*40}\nEvaluating Model {model_idx}: {model}")

        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X, y), start=1):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            val_preds = model.predict(X_val).reshape(-1, 1)
            oof_predictions[model_idx][val_idx] = val_preds

            val_df = train_df.iloc[val_idx]
            submission_df = pd.DataFrame({id_column: range(len(val_preds)), 'prediction': val_preds.flatten()})
            score = stratified_concordance(
                val_df[['efs', 'efs_time', 'race_group']].reset_index(),
                submission_df.copy(),
                id_column
            )
            model_scores[model_idx][fold_idx - 1] = score
            final_predictions[model_idx] += model.predict(X_test)

            print(f"Fold {fold_idx} | Concordance Score: {score:.4f}")

        final_predictions[model_idx] /= folds

    return oof_predictions, model_scores, final_predictions


In [34]:
def evaluate_survival_models(models, X_train, y_train, X_test, train_df, id_column, n_folds):

    oof_predictions = [np.zeros(len(X_train)) for _ in models]
    model_scores = [[] for _ in models]
    final_predictions = [np.zeros(len(X_test)) for _ in models]

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    for model_idx, model in enumerate(models):
        print(f"\n{'='*40}\nEvaluating Survival Model {model_idx}: {model}")

        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train), start=1):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            durations_train = train_df.iloc[train_idx]['efs_time'].values.astype(np.float32)
            events_train = train_df.iloc[train_idx]['efs'].values.astype(np.float32)
            durations_val = train_df.iloc[val_idx]['efs_time'].values.astype(np.float32)
            events_val = train_df.iloc[val_idx]['efs'].values.astype(np.float32)

            print(f"Fold {fold_idx}: val_idx shape = {val_idx.shape}, X_val shape = {X_val.shape}")

            X_tr_tensor = torch.tensor(X_tr, dtype=torch.float32)
            durations_train_tensor = torch.tensor(durations_train, dtype=torch.int64)
            events_train_tensor = torch.tensor(events_train, dtype=torch.float32)
            X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

            try:
                if isinstance(model, DeepHitSingle):
                    model.fit(X_tr_tensor, (durations_train_tensor, events_train_tensor),
                              batch_size=64, epochs=50, verbose=False)
                    val_preds = model.predict_surv_df(X_val_tensor).iloc[:, -1].values
                # elif isinstance(model, CoxPH):
                #   try:
                #       from torch.utils.data import TensorDataset, DataLoader

                #       durations_events_tensor = torch.cat(
                #           [durations_train_tensor.view(-1, 1), events_train_tensor.view(-1, 1)], dim=1
                #       )
                #       train_dataset = TensorDataset(X_tr_tensor, durations_train_tensor, events_train_tensor)

                #       train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

                #       for epoch in range(50):  # Specify number of epochs
                #           for batch_X, batch_durations, batch_events in train_loader:
                #               model.optimizer.zero_grad()  # Reset gradients
                #               log_h = model.net(batch_X)  # Forward pass
                #               loss = model.loss(log_h, batch_durations, batch_events)  # Compute loss
                #               loss.backward()  # Backpropagation
                #               model.optimizer.step()  # Update weights

                #       val_preds = model.predict_partial_hazard(X_val_tensor).squeeze()

                #   except RuntimeError as e:
                #       print(f"RuntimeError during training: {e}")
                #       raise

                elif isinstance(model, LGBMRegressor):
                    model.fit(X_tr, durations_train)
                    val_preds = model.predict(X_val)

                val_preds = np.asarray(val_preds).flatten()
                print(f"Model {model_idx}: val_preds shape = {val_preds.shape}")

                oof_predictions[model_idx][val_idx] = np.nan
                for idx, pred in zip(val_idx[:len(val_preds)], val_preds):
                    oof_predictions[model_idx][idx] = pred

                val_df = train_df.iloc[val_idx]
                submission_df = pd.DataFrame({id_column: range(len(val_preds)), 'prediction': val_preds})
                score = stratified_concordance(
                    val_df[['efs', 'efs_time', 'race_group']].reset_index(),
                    submission_df.copy(),
                    id_column
                )
                model_scores[model_idx].append(score)
                print(f"Fold {fold_idx} | Concordance Score: {score:.4f}")

            except ValueError as e:
                print(f"Error while fitting/predicting with model: {e}")
                raise

        if isinstance(model, (DeepHitSingle, CoxPH)):
            X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
            final_predictions[model_idx] = model.predict_surv_df(X_test_tensor).iloc[:, -1].values
        else:
            final_predictions[model_idx] = model.predict(X_test)

    return oof_predictions, model_scores, final_predictions

In [35]:
y_train_durations = train['efs_time'].values
y_train_events = train['efs'].values

y_train_combined = np.vstack((y_train_durations, y_train_events)).T

In [36]:
def hyperparameter_optimization(trial, model_type, X_train, y_train, X_test, train_df, id_column, num_folds=n_folds, random_seed=42):
    if model_type == "LGBM":
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
        }
        model = LGBMRegressor(**params, verbose=-1, metric=None, random_state=random_seed)
    elif model_type == "XGB":
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        }
        model = XGBRegressor(**params, enable_categorical=True, random_state=random_seed)
    elif model_type == "CatBoost":
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "depth": trial.suggest_int("depth", 4, 12),
            "iterations": trial.suggest_int("iterations", 100, 1000),
        }
        model = CatBoostRegressor(**params, silent=True, random_state=random_seed)
    elif model_type == "XGB_Cox":
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        }
        model = XGBRegressor(**params, objective='survival:cox', eval_metric='cox-nloglik', enable_categorical=True, random_state=random_seed)
    elif model_type == "CatBoost_Cox":
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "depth": trial.suggest_int("depth", 4, 12),
            "iterations": trial.suggest_int("iterations", 100, 1000),
        }
        model = CatBoostRegressor(**params, loss_function='Cox', silent=True, random_state=random_seed)
    elif model_type == "DeepSurv":
        num_nodes = [trial.suggest_int("num_nodes_layer1", 32, 128),
                     trial.suggest_int("num_nodes_layer2", 16, 64)]
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-2)
        duration_index = np.arange(y_train_combined[:, 0].max() + 1)

        out_features = len(duration_index)
        in_features = X_train.shape[1]

        net = MLPVanilla(in_features, num_nodes, out_features, batch_norm=True, dropout=dropout)
        model = DeepHitSingle(net, ttoptim.Adam(lr=lr), duration_index=duration_index)
    elif model_type == "GBM_Survival":
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
        }
        model = LGBMRegressor(**params, objective="regression", metric="rmse", random_state=random_seed)
    elif model_type == "DeepHit":
        num_nodes = [trial.suggest_int("num_nodes_layer1", 32, 128),
                     trial.suggest_int("num_nodes_layer2", 16, 64)]
        dropout = trial.suggest_float("dropout", 0.1, 0.5)
        lr = trial.suggest_float("learning_rate", 1e-4, 1e-2)
        duration_index = np.arange(y_train_combined[:, 0].max() + 1)

        out_features = len(duration_index)
        in_features = X_train.shape[1]

        net = MLPVanilla(in_features, num_nodes, out_features, batch_norm=True, dropout=dropout)
        model = DeepHitSingle(net, ttoptim.Adam(lr=lr), duration_index=duration_index)

    else:
        raise ValueError(f"Unsupported model type: {model_type}")

    kf = KFold(n_splits=num_folds, shuffle=True, random_state=random_seed)
    scores = []

    for train_idx, val_idx in kf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        if model_type == "DeepHit":
            durations_train = train_df.iloc[train_idx]['efs_time'].values.astype(np.float32)
            events_train = train_df.iloc[train_idx]['efs'].values.astype(np.float32)
            durations_val = train_df.iloc[val_idx]['efs_time'].values.astype(np.float32)
            events_val = train_df.iloc[val_idx]['efs'].values.astype(np.float32)

            X_tr_tensor = torch.tensor(X_tr.values, dtype=torch.float32)

            durations_train_tensor = torch.tensor(durations_train, dtype=torch.int64)
            events_train_tensor = torch.tensor(events_train, dtype=torch.float32)

            model.fit(X_tr_tensor, (durations_train_tensor, events_train_tensor), batch_size=64, epochs=50, verbose=False)

            X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32)

            val_predictions = model.predict_surv_df(X_val_tensor).iloc[:, -1].values
        else:
            model.fit(X_tr, y_tr)
            val_predictions = model.predict(X_val).reshape(-1, 1)

        val_df = train_df.iloc[val_idx]
        submission_df = pd.DataFrame({id_column: range(len(val_predictions)), "prediction": val_predictions.flatten()})
        score = stratified_concordance(
            val_df[['efs', 'efs_time', 'race_group']].reset_index(),
            submission_df.copy(),
            id_column
        )
        scores.append(score)

    return -np.mean(scores)

In [37]:
# # Hyperparameter Optimization for DeepHit
# study_deephit = optuna.create_study(direction="minimize")
# study_deephit.optimize(
#     partial(
#         hyperparameter_optimization,
#         model_type="DeepHit",
#         X_train=pd.DataFrame(X_train),
#         y_train=pd.Series(train['y']),
#         X_test=pd.DataFrame(X_test),
#         train_df=train,
#         id_column=id_column,
#         num_folds=n_folds,
#         random_seed=42
#     ),
#     n_trials=100
# )
# print("Best DeepHit Parameters:", study_deephit.best_params)


# # Hyperparameter Optimization for DeepSurv
# study_deepsurv = optuna.create_study(direction="minimize")
# study_deepsurv.optimize(
#     partial(
#         hyperparameter_optimization,
#         model_type="DeepSurv",
#         X_train=pd.DataFrame(X_train),
#         y_train=pd.Series(train['y']),
#         X_test=pd.DataFrame(X_test),
#         train_df=train,
#         id_column=id_column,
#         num_folds=n_folds,
#         random_seed=42
#     ),
#     n_trials=100
# )
# print("Best DeepSurv Parameters:", study_deepsurv.best_params)

# study_gbm_surv = optuna.create_study(direction="minimize")
# study_gbm_surv.optimize(
#     partial(
#         hyperparameter_optimization,
#         model_type="GBM_Survival",
#         X_train=pd.DataFrame(X_train),
#         y_train=pd.Series(train['y']),
#         X_test=pd.DataFrame(X_test),
#         train_df=train,
#         id_column=id_column,
#         num_folds=n_folds,
#         random_seed=42
#     ),
#     n_trials=100
# )
# print("Best GBM Survival Parameters:", study_gbm_surv.best_params)


In [38]:
# # Hyperparameter Optimization for Multiple Models
# study_xgb_cox = optuna.create_study(direction="minimize")
# study_xgb_cox.optimize(
#     partial(
#         hyperparameter_optimization,
#         model_type="XGB_Cox",
#         X_train=pd.DataFrame(X_train),
#         y_train=pd.Series(train['y']),
#         X_test=pd.DataFrame(X_test),
#         train_df=train,
#         id_column=id_column,
#         num_folds=5,
#         random_seed=42
#     ),
#     n_trials=100
# )
# print("Best XGBoost Cox Parameters:", study_xgb_cox.best_params)

# study_cat_cox = optuna.create_study(direction="minimize")
# study_cat_cox.optimize(
#     partial(
#         hyperparameter_optimization,
#         model_type="CatBoost_Cox",
#         X_train=pd.DataFrame(X_train),
#         y_train=pd.Series(train['y']),
#         X_test=pd.DataFrame(X_test),
#         train_df=train,
#         id_column=id_column,
#         num_folds=5,
#         random_seed=42
#     ),
#     n_trials=100
# )
# print("Best CatBoost Cox Parameters:", study_cat_cox.best_params)


In [39]:
#Best XGBoost Cox Parameters: {'learning_rate': 0.09288700457765192,
#'max_depth': 12, 'n_estimators': 830, 'subsample': 0.6911432280129449, 'colsample_bytree': 0.8073544074701855}
#Trial 50 finished with value: -0.34304094517333406 and parameters: {'learning_rate': 0.28658161126461923,
#'depth': 12, 'iterations': 454}. Best is trial 50 with value: -0.34304094517333406.
#Best GBM Survival Parameters: {'learning_rate': 0.02483481468939416, 'num_leaves': 36,
#'n_estimators': 688, 'max_depth': 14}
# DeepHit parameters: {'num_nodes_layer1': 79, 'num_nodes_layer2': 34, 'dropout': 0.30250948927624044,
#'learning_rate': 0.003862956035804037}
#DeepSurv parameters: {'num_nodes_layer1': 61, 'num_nodes_layer2': 18,
#'dropout': 0.2743469700755701, 'learning_rate': 0.0031077035853385128}
optimized_params_lgbm_kaplan = {
    'learning_rate': 0.09025302904716048,
    'num_leaves': 128,
    'n_estimators': 378,
    'max_depth': 3
}
optimized_params_xgb_kaplan = {
    'learning_rate': 0.030125743801830442,
    'max_depth': 4,
    'n_estimators': 734,
    'subsample': 0.7512601957879981,
    'colsample_bytree': 0.6747072457305869
}
optimized_params_cat_kaplan = {
    'learning_rate': 0.028120567128727205,
    'depth': 7,
    'iterations': 960
}
optimized_params_xgb_cox = {
    'learning_rate': 0.09288700457765192,
    'max_depth': 12,
    'n_estimators': 830,
    'subsample': 0.6911432280129449,
    'colsample_bytree': 0.8073544074701855
}
optimized_params_cat_cox = {
    'learning_rate': 0.28365953784848147,
    'depth': 12,
    'iterations': 811
}
optimized_params_gbm = {
    'learning_rate': 0.02483481468939416,
    'num_leaves': 36,
    'n_estimators': 688,
    'max_depth': 14
}
optimized_params_deepHit = {
    'num_nodes_layer1': 79,
    'num_nodes_layer2': 34,
    'dropout': 0.30250948927624044,
    'learning_rate': 0.003862956035804037
}
optimized_params_deepSurv = {
    'num_nodes_layer1': 61,
    'num_nodes_layer2': 18,
    'dropout': 0.2743469700755701,
    'learning_rate': 0.0031077035853385128
}

In [40]:
lgbm_model_kaplan = LGBMRegressor(
    **optimized_params_lgbm_kaplan,
    verbose=-1,
    metric=None,
    random_state=42
)
xgb_model_kaplan = XGBRegressor(
    **optimized_params_xgb_kaplan,
    enable_categorical=True,
    random_state=42
)
catboost_model_kaplan = CatBoostRegressor(
    **optimized_params_cat_kaplan,
    silent=True,
    random_state=42
)
xgb_model_cox = XGBRegressor(
    **optimized_params_xgb_cox,
    objective='survival:cox',
    eval_metric='cox-nloglik',
    enable_categorical=True,
    random_state=42
)
catboost_model_cox = CatBoostRegressor(
    **optimized_params_cat_cox,
    loss_function='Cox',
    silent=True,
    random_state=42
)

gbm_survival_model = LGBMRegressor(
    **optimized_params_gbm,
    objective="regression",
    metric="rmse",
    random_state=42
)

deephit_model = DeepHitSingle(
    MLPVanilla(
        in_features=X_train.shape[1],
        num_nodes=[optimized_params_deepHit['num_nodes_layer1'], optimized_params_deepHit['num_nodes_layer2']],
        out_features = len(np.arange(y_train_combined[:, 0].max() + 1)),
        batch_norm=True,
        dropout=optimized_params_deepHit['dropout']
    ),
    ttoptim.Adam(lr=optimized_params_deepHit['learning_rate']),
    duration_index = np.arange(y_train_combined[:, 0].max() + 1)
)

deepsurv_model = CoxPH(
    MLPVanilla(
        in_features=X_train.shape[1],
        num_nodes=[optimized_params_deepSurv['num_nodes_layer1'], optimized_params_deepSurv['num_nodes_layer2']],
        out_features = len(np.arange(y_train_combined[:, 0].max() + 1)),
        batch_norm=True,
        dropout=optimized_params_deepSurv['dropout']
    ),
    ttoptim.Adam(lr=optimized_params_deepSurv['learning_rate']),
    device='cpu',
)

survival_models = [
    gbm_survival_model,
    deephit_model
]

kaplan_models = [
    lgbm_model_kaplan, xgb_model_kaplan, catboost_model_kaplan
]
cox_models = [
    xgb_model_cox, catboost_model_cox
]

In [41]:
survival_oof_predictions, survival_model_scores, survival_final_predictions = evaluate_survival_models(
    survival_models,
    X_train,
    y_train,
    X_test,
    train,
    id_column,
    n_folds
)


Evaluating Survival Model 0: LGBMRegressor(learning_rate=0.02483481468939416, max_depth=14, metric='rmse',
              n_estimators=688, num_leaves=36, objective='regression',
              random_state=42)
Fold 1: val_idx shape = (1920,), X_val shape = (1920, 213)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010840 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1007
[LightGBM] [Info] Number of data points in the train set: 26880, number of used features: 210
[LightGBM] [Info] Start training from score 23.157448
Model 0: val_preds shape = (1920,)
Fold 1 | Concordance Score: 0.3717
Fold 2: val_idx shape = (1920,), X_val shape = (1920, 213)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011455 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you

In [42]:
kaplan_oof_predictions, kaplan_model_scores, kaplan_final_predictions = model_evaluation(
    kaplan_models,
    X_train,
    y_train,
    X_test,
    train,
    id_column,
    n_folds
)


Evaluating Model 0: LGBMRegressor(learning_rate=0.09025302904716048, max_depth=3, metric=None,
              n_estimators=378, num_leaves=128, random_state=42, verbose=-1)
Fold 1 | Concordance Score: 0.6579
Fold 2 | Concordance Score: 0.6680
Fold 3 | Concordance Score: 0.6636
Fold 4 | Concordance Score: 0.6679
Fold 5 | Concordance Score: 0.6750
Fold 6 | Concordance Score: 0.6581
Fold 7 | Concordance Score: 0.6562
Fold 8 | Concordance Score: 0.6597
Fold 9 | Concordance Score: 0.6568
Fold 10 | Concordance Score: 0.6429
Fold 11 | Concordance Score: 0.6506
Fold 12 | Concordance Score: 0.6570
Fold 13 | Concordance Score: 0.6499
Fold 14 | Concordance Score: 0.6485
Fold 15 | Concordance Score: 0.6598

Evaluating Model 1: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6747072457305869, device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric=N

In [43]:
y_efs_time_new = train['efs_time_new']
y_efs_time_new

0        -42.356
1          4.672
2        -19.793
3       -102.349
4        -16.223
          ...   
28795    -18.633
28796      4.892
28797    -23.157
28798    -52.351
28799    -25.158
Name: efs_time_new, Length: 28800, dtype: float64

In [44]:
cox_oof_predictions, cox_model_scores, cox_final_predictions = model_evaluation(
    cox_models,
    X_train,
    y_efs_time_new,
    X_test,
    train,
    id_column,
    n_folds
)


Evaluating Model 0: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8073544074701855, device=None,
             early_stopping_rounds=None, enable_categorical=True,
             eval_metric='cox-nloglik', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.09288700457765192, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=12, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=830, n_jobs=None,
             num_parallel_tree=None, ...)
Fold 1 | Concordance Score: 0.4674
Fold 2 | Concordance Score: 0.4610
Fold 3 | Concordance Score: 0.4357
Fold 4 | Concordance Score: 0.6417
Fold 5 | Concordance Score: 0

In [45]:
oof_predictions_total = []
oof_predictions_total += kaplan_oof_predictions
oof_predictions_total += cox_oof_predictions
oof_predictions_total += survival_oof_predictions
print(oof_predictions_total)

[array([[0.5024296 ],
       [0.64438491],
       [0.45870815],
       ...,
       [0.69283391],
       [0.56338392],
       [0.48649209]]), array([[0.50060523],
       [0.65050709],
       [0.45433322],
       ...,
       [0.71406925],
       [0.55056137],
       [0.47662765]]), array([[0.50208188],
       [0.66632485],
       [0.451546  ],
       ...,
       [0.68977398],
       [0.55643483],
       [0.50118365]]), array([[4.39401716e-02],
       [5.75893473e+17],
       [           inf],
       ...,
       [5.70742941e+00],
       [1.38489142e-01],
       [2.23365724e-02]]), array([[-2.73420101],
       [ 1.53984283],
       [-5.62068595],
       ...,
       [-0.84073811],
       [-0.53185486],
       [-2.43133646]]), array([37.92972563, 20.01470557, 18.01319459, ...,  9.737711  ,
       24.11419451, 22.7727863 ]), array([0.99983245, 0.99916738, 0.99998283, ...,        nan,        nan,
              nan])]


In [46]:
final_ranked_predictions = 0

for i, prediction in enumerate(oof_predictions_total):
    final_ranked_predictions += rankdata(prediction)

final_ranked_predictions

array([nan, nan, nan, ..., nan, nan, nan])

In [47]:
total_predictions = []
total_predictions += kaplan_final_predictions
total_predictions += cox_final_predictions
total_predictions += survival_final_predictions
total_predictions

[array([0.50436222, 0.63374852, 0.4571218 ]),
 array([0.50454024, 0.66246169, 0.4521884 ]),
 array([0.5021584 , 0.6744766 , 0.45469925]),
 array([inf, inf, inf]),
 array([-3.24033729,  3.45946874, -5.95515874]),
 array([38.43050064, 30.45530726, 17.17369452]),
 array([0.99964565, 0.9958243 , 0.97389925, 0.94085246, 0.8567717 ,
        0.7788949 , 0.740309  , 0.730642  , 0.72833866, 0.72654605,
        0.72614354, 0.72579175, 0.7257037 , 0.72562003, 0.7255953 ,
        0.7255634 , 0.7255364 , 0.72552896, 0.72552574, 0.7255193 ,
        0.7255163 , 0.72551036, 0.72549725, 0.72549516, 0.7254951 ,
        0.7254951 , 0.725492  , 0.72549134, 0.7254902 , 0.7254539 ,
        0.7254539 , 0.7254539 , 0.7254472 , 0.72544706, 0.725423  ,
        0.725423  , 0.725423  , 0.72542286, 0.72542286, 0.72542286,
        0.72542274, 0.7254226 , 0.7254226 , 0.72542226, 0.72542226,
        0.72542083, 0.72542083, 0.72542083, 0.72542083, 0.72542083,
        0.72542083, 0.72542083, 0.7254208 , 0.7254208 , 0.7

In [48]:
# total_predictions = total_predictions[8:11]

# total_ranked_predictions = np.zeros_like(total_predictions[0])

# for prediction in total_predictions:
#     ranked = rankdata(prediction)
#     if ranked.shape != total_ranked_predictions.shape:
#         raise ValueError(f"Shape mismatch: ranked {ranked.shape}, expected {total_ranked_predictions.shape}")
#     total_ranked_predictions += ranked

# print(total_ranked_predictions)
np.random.seed(493) 
random_indices = np.random.choice(len(total_predictions), size=3, replace=False)
random_predictions = [total_predictions[i] for i in random_indices]

total_ranked_predictions = np.zeros_like(random_predictions[0])

for prediction in random_predictions:
    ranked = rankdata(prediction)
    if ranked.shape != total_ranked_predictions.shape:
        raise ValueError(f"Shape mismatch: ranked {ranked.shape}, expected {total_ranked_predictions.shape}")
    total_ranked_predictions += ranked

print("Selected random predictions indices:", random_indices)
print("Total ranked predictions:", total_ranked_predictions)

Selected random predictions indices: [3 4 1]
Total ranked predictions: [6. 8. 4.]


In [49]:
submission = pd.read_csv(sub_path)
submission['prediction'] = total_ranked_predictions
submission.to_csv('submission.csv', index=False)

print(submission.shape)
print(submission.head())

(3, 2)
      ID  prediction
0  28800         6.0
1  28801         8.0
2  28802         4.0
